# Arcadia run 2: BPE tokenizer on a Kaggle T4

Same model as run 1 (6 layers, 6 heads, 384 wide, context 256), but with an 8,000-token
SentencePiece BPE tokenizer trained on our own corpus, per-tier validation loss, and
bits-per-character so it can be compared with run 1 (1.776 bits/char).

Before running: **Settings → Accelerator → GPU T4**, **Internet on**, and **Add-ons → Secrets →
`HF_TOKEN`** ticked (only needed for the upload in the last cell).

For the real run use **Save Version → Save & Run All**, so it keeps going if you close the tab.

## 1. Clone the repo

In [ ]:
REPO = "https://github.com/DanielGregory/Arcadia"
BRANCH = "claude/arcadia-ml-setup-hhn3dv"

import os
if not os.path.exists("/kaggle/working/Arcadia"):
    !git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/Arcadia
%cd /kaggle/working/Arcadia
!git log --oneline -1

## 2. Install only what Kaggle is missing (torch is never reinstalled)

In [ ]:
import importlib.util, subprocess, sys

missing = [pkg for pkg in ("sentencepiece", "datasets") if importlib.util.find_spec(pkg) is None]
print("installing:", missing or "nothing, all present")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 3. Check the GPU

In [ ]:
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: set Accelerator to GPU T4")

## 4. Build the corpus

Also writes `val_core.txt`, `val_dream.txt`, `val_seasoning.txt` and `val_dreambank.txt` for the per-tier loss.

In [ ]:
!python -m engine.data_prep.build_corpus

## 5. Train the tokenizer

From `train.txt` only, so the validation text stays unseen. Takes a few seconds and checks that every val file round-trips exactly.

In [ ]:
!python -m engine.tokenizer --vocab_size 8000

## 6. Train

Settings are in `configs/run2.py`. Each eval line shows the loss per token, bits per character
(overall and per tier), the learning rate and elapsed time. Early stopping ends the run if val
loss hasn't improved for 4 evaluations; `best.pt` always keeps the best one.

In [ ]:
!python -m engine.train --config configs/run2.py --out_dir /kaggle/working/run2

## 7. Samples

Three whole dreams at each temperature (each starts at `<dream>` and stops at `</dream>`), then 500 tokens of free text from a blank line.

In [ ]:
!python -m engine.sample --ckpt /kaggle/working/run2/best.pt --dreams 3 --tokens 400 --temperature 0.7 0.9 1.1
!python -m engine.sample --ckpt /kaggle/working/run2/best.pt --tokens 500 --temperature 0.9

## 8. Upload to Hugging Face

Goes to `DannyGregs/arcadia-dreams/run2` (private). Skips if the `HF_TOKEN` secret isn't attached.

In [ ]:
HF_REPO = "DannyGregs/arcadia-dreams"
RUN_DIR = "/kaggle/working/run2"

import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as err:
    print("no HF_TOKEN secret, skipping upload:", err)
else:
    !python -m engine.hub push --repo {HF_REPO} --run_dir {RUN_DIR}